In [1]:
import pandas as pd
from sqlalchemy import create_engine


In [2]:
USERNAME = "postgres"
PASSWORD = "admin"
HOST = "localhost"
PORT = "5432"
DATABASE = "ecommerce_database"


In [3]:
engine = create_engine(
    f"postgresql+psycopg2://{USERNAME}:{PASSWORD}@{HOST}:{PORT}/{DATABASE}"
)


In [4]:
files = {
    "product_category_name_translation": "product_category_name_translation.csv",
    "geolocation": "geolocation.csv",
    "customers": "customers.csv",
    "sellers": "sellers.csv",
    "products": "products.csv",
    "orders": "orders.csv",
    "order_items": "order_items.csv",
    "order_payments": "order_payments.csv",
    "order_reviews": "order_reviews.csv"
}


In [5]:
create_tables_sql = """

CREATE SCHEMA IF NOT EXISTS olist;

SET search_path TO olist;

DROP TABLE IF EXISTS order_reviews CASCADE;
DROP TABLE IF EXISTS order_payments CASCADE;
DROP TABLE IF EXISTS order_items CASCADE;
DROP TABLE IF EXISTS orders CASCADE;
DROP TABLE IF EXISTS products CASCADE;
DROP TABLE IF EXISTS sellers CASCADE;
DROP TABLE IF EXISTS customers CASCADE;
DROP TABLE IF EXISTS geolocation CASCADE;
DROP TABLE IF EXISTS product_category_name_translation CASCADE;

-- =========================================
-- product_category_name_translation
-- =========================================
CREATE TABLE product_category_name_translation (
    product_category_name VARCHAR(100) PRIMARY KEY,
    product_category_name_english VARCHAR(100)
);

-- =========================================
-- geolocation
-- =========================================
CREATE TABLE geolocation (
    geolocation_zip_code_prefix INTEGER,
    geolocation_lat NUMERIC(10,6),
    geolocation_lng NUMERIC(10,6),
    geolocation_city VARCHAR(100),
    geolocation_state CHAR(2)
);

-- =========================================
-- customers
-- =========================================
CREATE TABLE customers (
    customer_id VARCHAR(50) PRIMARY KEY,
    customer_unique_id VARCHAR(50),
    customer_zip_code_prefix INTEGER,
    customer_city VARCHAR(100),
    customer_state CHAR(2)
);

-- =========================================
-- sellers
-- =========================================
CREATE TABLE sellers (
    seller_id VARCHAR(50) PRIMARY KEY,
    seller_zip_code_prefix INTEGER,
    seller_city VARCHAR(100),
    seller_state CHAR(2)
);

-- =========================================
-- products
-- =========================================
CREATE TABLE products (
    product_id VARCHAR(50) PRIMARY KEY,
    product_category_name VARCHAR(100),

    product_name_lenght INTEGER,
    product_description_lenght INTEGER,
    product_photos_qty INTEGER,
    product_weight_g INTEGER,
    product_length_cm INTEGER,
    product_height_cm INTEGER,
    product_width_cm INTEGER

);

-- =========================================
-- orders
-- =========================================
CREATE TABLE orders (
    order_id VARCHAR(50) PRIMARY KEY,
    customer_id VARCHAR(50),

    order_status VARCHAR(20),
    order_purchase_timestamp TIMESTAMP,
    order_approved_at TIMESTAMP,
    order_delivered_carrier_date TIMESTAMP,
    order_delivered_customer_date TIMESTAMP,
    order_estimated_delivery_date TIMESTAMP,

    FOREIGN KEY (customer_id)
    REFERENCES customers(customer_id)
);

-- =========================================
-- order_items
-- =========================================
CREATE TABLE order_items (
    order_id VARCHAR(50),
    order_item_id INTEGER,
    product_id VARCHAR(50),
    seller_id VARCHAR(50),

    shipping_limit_date TIMESTAMP,
    price NUMERIC(10,2),
    freight_value NUMERIC(10,2),

    PRIMARY KEY (order_id, order_item_id),

    FOREIGN KEY (order_id)
    REFERENCES orders(order_id),

    FOREIGN KEY (product_id)
    REFERENCES products(product_id),

    FOREIGN KEY (seller_id)
    REFERENCES sellers(seller_id)
);

-- =========================================
-- order_payments
-- =========================================
CREATE TABLE order_payments (
    order_id VARCHAR(50),
    payment_sequential INTEGER,

    payment_type VARCHAR(30),
    payment_installments INTEGER,
    payment_value NUMERIC(10,2),

    PRIMARY KEY (order_id, payment_sequential),

    FOREIGN KEY (order_id)
    REFERENCES orders(order_id)
);

-- =========================================
-- order_reviews
-- =========================================
CREATE TABLE order_reviews (
    review_id VARCHAR(50),
    order_id VARCHAR(50),

    review_score INTEGER,
    review_comment_title TEXT,
    review_comment_message TEXT,
    review_creation_date TIMESTAMP,
    review_answer_timestamp TIMESTAMP,

    PRIMARY KEY (review_id, order_id),

    FOREIGN KEY (order_id)
    REFERENCES orders(order_id)
);

"""

In [6]:
conn = engine.raw_connection()
cursor = conn.cursor()

cursor.execute(create_tables_sql)

conn.commit()

cursor.close()
conn.close()


In [7]:

print("Tables created successfully!")

load_order = [
    "product_category_name_translation",
    "geolocation",
    "customers",
    "sellers",
    "products",
    "orders",
    "order_items",
    "order_payments",
    "order_reviews"
]

Tables created successfully!


In [8]:
for table in load_order:

    file_path = files[table]

    print(f"Loading {table}...")

    df = pd.read_csv('data/'+file_path)

    df.to_sql(
        name=table,
        con=engine,
        schema="olist",
        if_exists="append",
        index=False
    )

    print(f"{table} loaded successfully!")

print("All tables loaded successfully!")

Loading product_category_name_translation...
product_category_name_translation loaded successfully!
Loading geolocation...
geolocation loaded successfully!
Loading customers...
customers loaded successfully!
Loading sellers...
sellers loaded successfully!
Loading products...
products loaded successfully!
Loading orders...
orders loaded successfully!
Loading order_items...
order_items loaded successfully!
Loading order_payments...
order_payments loaded successfully!
Loading order_reviews...
order_reviews loaded successfully!
All tables loaded successfully!
